<a href="https://colab.research.google.com/github/MohanGaikwad/hyderabad-iot-event/blob/main/MobileNet_BYOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installation

In [ ]:
!pip install qai_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.4 MB/s eta 0:00:00


## Load Model from TorchVision

In [ ]:
import torch
from torchvision.models import mobilenet_v2

In [ ]:
# Using pre-trained MobileNet
torch_model = mobilenet_v2(pretrained=True)
torch_model.eval()

# Trace model (for on-device deployment)
input_shape = (1, 3, 224, 224)
example_input = torch.rand(input_shape)
torch_model = torch.jit.trace(torch_model, example_input)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 32.4MB/s]


## Configure AI Hub
### Get API Token from AI Hub

1. Go to https://app.aihub.qualcomm.com/
2. Your Account -> Setting -> Account
2. Copy API token and configure your account

In [ ]:
!qai-hub configure --api_token <>

## Select Device

In [ ]:
!qai-hub list-devices

+-------------------------------+------------+----------+---------+---------------------------------------+----------------------------------------------------------+
|             Device            |     OS     |  Vendor  |   Type  |                Chipset                |                      CLI Invocation                      |
+-------------------------------+------------+----------+---------+---------------------------------------+----------------------------------------------------------+
|    Apple iPhone 15 Pro Max    | Ios 17.3.1 |  Apple   |  Phone  |               apple-a17               |  --device "Apple iPhone 15 Pro Max" --device-os 17.3.1   |
|        Apple iPhone 15        | Ios 17.2.1 |  Apple   |  Phone  |               apple-a16               |      --device "Apple iPhone 15" --device-os 17.2.1       |
|        Apple iPhone 14        |  Ios 16.1  |  Apple   |  Phone  |               apple-a15               |       --device "Apple iPhone 14" --device-os 16.1        

In [ ]:
import qai_hub as hub
device = hub.Device("Snapdragon 8 Elite QRD")

## Compile model for device

In [ ]:
runtime="--target_runtime tflite" # TensorFlow Lite
#runtime="--target_runtime onnx" # ONNX

# Optimize model for the chosen device
compile_job = hub.submit_compile_job(
    model=torch_model,
    device=device,
    input_specs=dict(image=input_shape),
    options=runtime,
)

Uploading model: 100%|██████████| 13.9M/13.9M [00:01<00:00, 14.4MB/s]


Scheduled compile job (jz57d2j95) successfully. To see the status and results:
    https://app.aihub.qualcomm.com/jobs/jz57d2j95/



## Run model on device and performance evaluation

In [ ]:
target_model = compile_job.get_target_model()
profile_job = hub.submit_profile_job(
                   model=target_model,
                   device=device,
)

Waiting for compile job (jz57d2j95) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Scheduled profiling job (jqp4wnx1g) successfully. To see the status and results:
    https://app.aihub.qualcomm.com/jobs/jqp4wnx1g/



## Run inference on-device
Run inference on specified device with given input data

In [ ]:
import numpy as np

sample = np.random.random((1, 3, 224, 224)).astype(np.float32)

inference_job = hub.submit_inference_job(
    model=compile_job.get_target_model(),
    device=device,
    inputs=dict(image=[sample])
)

assert isinstance(inference_job, hub.InferenceJob)
on_device_output = inference_job.download_output_data()


Uploading dataset: 100%|██████████| 541k/541k [00:00<00:00, 1.06MB/s]


Scheduled inference job (j0px197lg) successfully. To see the status and results:
    https://app.aihub.qualcomm.com/jobs/j0px197lg/

Waiting for inference job (j0px197lg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


dataset-d82ne8ko2.h5: 100%|██████████| 14.5k/14.5k [00:00<00:00, 4.14MB/s]


In [ ]:
# Helper methods for accuracy check
from math import log10, sqrt
import numpy as np

def get_psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    if(mse == 0):
        return 100
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    return psnr


## Numerical evaluation to check on-device model accuracy

In [ ]:
# Run torch model
torch_output = torch_model(torch.Tensor(sample))
# Compare Torch vs on-device output
print("PSNR: ", get_psnr(torch_output.detach().numpy(), on_device_output['output_0'][0]))

PSNR:  91.08814300785914
